In [2]:
import sys
import os
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir      = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config_loader import load_config
import triage_filter

cfg = load_config()

# Resolve triage path (new key — created by run_triage if absent)
triage_path = Path(cfg["paths"]["triage"])

print("Config loaded.")
print("  raw_images :", cfg["paths"]["raw_images"])
print("  triage     :", triage_path)


Config loaded.
  raw_images : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw
  triage     : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/triage


# Stage 00a.2 — SigLIP Triage Filter

**What this notebook does:**  
Scores every raw patent image with [SigLIP](https://huggingface.co/timm/ViT-SO400M-14-SigLIP-384) (a large vision-language model) to detect images that are **not technical drawings** — tables, text pages, data grids, title sheets — before those images enter the expensive labelling pipeline downstream.

It is **completely non-destructive**: nothing in `raw/` is touched. Outputs are JSON/CSV manifests only.

---

## Why this step exists

PatSeer downloads *everything* on the Drawings tab — which often includes:
- Cover pages / title sheets
- Tables of component labels
- Text-only description pages
- Flowcharts and block diagrams that look like drawings but aren't figure crops

Running `00b` (figure crop + label matching) on those wastes GPU time and pollutes the labelled dataset. This step flags them so Stage `00b` can skip them.

---

## How the scoring works

Each image is scored against two zero-shot text prompts using SigLIP:

| Prompt | Meaning |
|--------|---------|
| `"a technical patent line drawing of an aircraft or mechanical component"` | → `score_drawing` |
| `"a table, chart, text page, or data grid with no technical drawing"` | → `score_table` |

The two scores are softmax-normalised so they sum to 1.  
`keep = True` when `score_drawing ≥ threshold` (default **0.60**).  
`keep = False` = flagged as non-drawing → will be skipped by `00b`.

> **Threshold guide:**  
> `0.60` (default) — conservative, keeps borderline images, low false-positive rate  
> `0.65` — moderate filtering  
> `0.70` — aggressive, only keep high-confidence drawings

---

## Outputs

Both written under `cfg["paths"]["triage"]` = `1639_DS/triage/`:

| File | Content |
|------|---------|
| `<patent_id>.json` | Per-image scores for one patent: `file`, `pred`, `score_drawing`, `score_table`, `keep` |
| `triage_summary.csv` | One row per patent: `patent_id`, `total_images`, `flagged_count`, `flagged_ratio` |

Already-processed patents are **not re-scored** on repeat runs (JSON exists → skip).

---

## Where it fits in the pipeline

```
00a    (download from PatSeer)
 ↓
00a.1  (audit: which patents are missing / need re-download)
 ↓
00a.2  ← YOU ARE HERE  (score every image: drawing vs. non-drawing)
 ↓
00b    (figure crop + _F/_Fu label matching — uses triage to skip flagged images)
 ↓
02     (pad + resize to 518×518)
```

---

## Cell guide

| Cell | What it does |
|------|--------------|
| 1 | Load config — resolves `raw_images` and `triage` paths |
| 2 | Load SigLIP model (~3 GB weights, cached after first run, needs GPU) |
| 3 | Run triage — set `limit=10` to test on 10 patents first, `limit=None` for full dataset |
| 4 | Inspect results — flagged ratio summary + per-image table for worst patents |

---

## Before you run

- **GPU required** — SigLIP ViT-SO400M is large; CPU inference is very slow (~10 s/image).  
- **First run downloads ~3 GB** of weights into the HuggingFace cache (`~/.cache/huggingface/`).  
- Run Cell 3 with `limit=10` first to check the threshold is sensible for your data before processing all 1350 folders.  
- If scores cluster around 0.50–0.52 (as in the test output above), the prompts may need tuning for your specific patent images — see `src/triage_filter.py` → `_PROMPT_KEEP` / `_PROMPT_FLAG`.

In [2]:
# Loads SigLIP ViT-SO400M-14-SigLIP-384 via open_clip.
# Downloads ~3 GB of weights on first run (cached by HuggingFace Hub).
model, processor, device = triage_filter.load_siglip_model(cfg)
print(f"SigLIP ready on: {device}")


[SigLIP] Cache: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/models/SigLIP
[SigLIP] Loaded fp16 on cuda:0  (6.8 GB VRAM remaining)
SigLIP ready on: cuda:0


In [ ]:
# ── Run options ──────────────────────────────────────────────────────────────
#
# This cell runs SigLIP scoring — slow, and only needed ONCE per patent (or
# when new patent folders are added). Already-triaged patents are skipped by
# default so locked review decisions are never touched.
#
# To just change the keep/discard threshold on already-scored images, do NOT
# run this cell — use Cell 3b below instead (rethreshold_existing / reset_threshold),
# which is instant and never re-runs the model.
#
TEST_MODE  = False   # True = only 10 patents (quick sanity check)
                    # False = full dataset (all ~1614 folders)
THRESHOLD  = 0.53   # kept for API compatibility; logic now uses pure argmax
                    # (discard only when model prefers the table/text prompt)
FORCE      = False  # True = re-score and OVERWRITE already-triaged patents,
                    # destroying any locked review decisions. Only use this
                    # intentionally (e.g. raw images changed).
# ─────────────────────────────────────────────────────────────────────────────

limit = 600 if TEST_MODE else None
print(f"Running triage: {'TEST MODE (10 patents)' if TEST_MODE else 'FULL DATASET'}, threshold={THRESHOLD}")
triage_filter.run_triage(cfg, threshold=THRESHOLD, limit=limit, force=FORCE)

Running triage: FULL DATASET, threshold=0.53
[SigLIP] Cache: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/models/SigLIP
[SigLIP] Loaded fp16 on cuda:1  (6.4 GB VRAM remaining)
  ✓ AT503689A1_38777771  total=2  flagged=2
  ✓ AU2020100605A4_70776151  total=12  flagged=12
  ✓ AU2021232803A1_85775850  total=10  flagged=10
  ✓ AU2023270571A1_95939416  total=4  flagged=4
  ✓ BR112023017877B1_83067308  total=7  flagged=7
  ✓ CA2869734A1_49882410  total=10  flagged=9
  ✓ CA2874341A1_50068674  total=10  flagged=10
  ✓ CA2947683A1_62068329  total=5  flagged=5
  ✓ CA2958445A1_63166000  total=5  flagged=5
  ✓ CA2963502A1_63709012  total=2  flagged=2
  ✓ CA2967221A1_58707420  total=11  flagged=11
  ✓ CA2967228A1_58707421  total=11  flagged=11
  ✓ CA2967402A1_58709316  total=12  flagged=12
  ✓ CA3096221A1_81206876  total=41  flagged=40
  ✓ CN101885295A_43071443  total=12  flagged=11
  ✓ CN103786881A_50663046  total=12  flagged=12
  ✓ CN104229137A_52218225  total=9  flagged=9
  ✓

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  ✓ DE102019006484B3_71615573  total=12  flagged=12
  ✓ DE102019007020A1_75156016  total=12  flagged=12
  ✓ DE102019009187A1_73747659  total=1  flagged=1
  ✓ DE102019101358A1_71402602  total=1  flagged=1
  ✓ DE102019101359A1_71403069  total=4  flagged=4
  ✓ DE102019101362A1_71402646  total=3  flagged=3
  ✓ DE102019102189A1_67224462  total=5  flagged=4
  ✓ DE102019118023B3_71402838  total=5  flagged=5
  ✓ DE102019118024A1_74092679  total=2  flagged=2
  ✓ DE102019121788A1_74239987  total=2  flagged=2
  ✓ DE102020007834A1_81847393  total=8  flagged=8
  ✓ DE102020007836A1_81847214  total=8  flagged=8
  ✓ DE102020107437A1_77552521  total=3  flagged=3
  ✓ DE102020113488A1_78408629  total=12  flagged=12
  ✓ DE102020118674A1_79020761  total=3  flagged=3
  ✓ DE102020118677A1_79020751  total=2  flagged=2
  ✓ DE102020202612A1_73790105  total=4  flagged=4
  ✓ DE102021113538A1_81851524  total=3  flagged=3
  ✓ DE102023108565B3_89508163  total=6  flagged=6
  ✓ DE102023122141A1_94391001  total=2  flag

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  ✓ EP3290334A1_59745849  total=12  flagged=12
  ✓ EP3360780A1_61132051  total=12  flagged=10
  ✓ EP3453616A1_63293968  total=12  flagged=11
  ✓ EP3594113A1_63449420  total=10  flagged=10
  ✓ EP3636545A1_68242353  total=12  flagged=12
  ✓ EP3770063A1_67851083  total=8  flagged=8
  ✓ EP3805100A1_68242276  total=8  flagged=4


...

  ✓ WO2023211639A1_88519502  total=12  flagged=12
  ✓ WO2023272353A1_84689755  total=12  flagged=12


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  ✓ WO2024013392A1_87418680  total=5  flagged=5


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  ✓ WO2024020362A1_89618566  total=10  flagged=10
  ✓ WO2024076966A1_88689476  total=12  flagged=12
  ✓ WO2024151705A1_89977947  total=7  flagged=6
  ✓ WO2024153885A1_86007350  total=6  flagged=6
  ✓ WO2024153886A1_86007085  total=6  flagged=6
  ✓ WO2024154415A1_91955601  total=1  flagged=1
  ✓ WO2024164048A1_92261759  total=8  flagged=7
  ✓ WO2024168389A1_92421352  total=10  flagged=10
  ✓ WO2024178469A1_92589006  total=8  flagged=8
  ✓ WO2024182866A1_92673862  total=6  flagged=6
  ✓ WO2024191152A1_92755439  total=6  flagged=6
  ✓ WO2024227493A1_93334018  total=3  flagged=3
  ✓ WO2024233261A2_93430973  total=7  flagged=7
  ✓ WO2024233650A1_91585552  total=12  flagged=12
  ✓ WO2024246895A1_93656806  total=12  flagged=12
  ✓ WO2024252755A1_93795837  total=12  flagged=12
  ✓ WO2025014192A1_94215603  total=12  flagged=12
  ✓ WO2025065072A1_95203924  total=12  flagged=9
  ✓ WO2025106962A1_95743549  total=4  flagged=4
  ✓ WO2025109312A1_93741523  total=8  flagged=7
  ✓ WO2025116483A1_958969

## Cell 3b — Re-threshold existing results (no GPU needed)

Already-confirmed discards (`keep=False`) are **never** upgraded back.  
Only images currently `keep=True` are re-evaluated against the new threshold.  
Rewrites all JSONs and `triage_summary.csv` in-place.

In [4]:
import importlib, triage_filter
importlib.reload(triage_filter)   # pick up src edits without restarting kernel

# ── Choose mode ───────────────────────────────────────────────────────────────
#
#   "add"   — rethreshold_existing: only ADDS new discards, never re-enables
#             anything already marked keep=False (safe, one-way)
#
#   "reset" — reset_threshold: re-derives keep from raw scores for ALL images,
#             works in both directions — use this to undo an over-aggressive run
#
MODE      = "reset"   # "add" | "reset"
THRESHOLD = 0.51      # tune this valuel
# ─────────────────────────────────────────────────────────────────────────────

if MODE == "add":
    triage_filter.rethreshold_existing(cfg, new_threshold=THRESHOLD)
elif MODE == "reset":
    triage_filter.reset_threshold(cfg, threshold=THRESHOLD)
else:
    print(f"Unknown MODE={MODE!r} — choose 'add' or 'reset'")


Reset complete (threshold=0.51): 13835 images | 3418 flagged (24.7%)
Summary rewritten: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/triage/triage_summary.csv


## Cell 5b — Lock/unlock (advanced)

Use **only** if you need direct control over locks outside the normal review flow.

| Action | Effect |
|--------|--------|
| `lock_discards` | Lock all current DISCARDs — they won't reappear in the viewer |
| `lock_keeps` | Lock all current KEEPs — aggressive threshold won't remove them |
| `unlock_all` | Clear every lock (full clean slate) |
| `stats` | Show current lock counts (no writes) |

> **Normal workflow:** use the Cell 6 viewer + the Commit cell at the bottom instead.

In [5]:
import importlib, triage_filter
importlib.reload(triage_filter)

# ── Choose lock action ────────────────────────────────────────────────────────
#
#   "lock_discards"  — protect current DISCARDs; threshold raises cannot re-enable them
#   "lock_keeps"     — protect current KEEPs; threshold raises cannot discard them
#   "unlock_all"     — clear all locks (clean slate before a full reset)
#   "stats"          — just print the current lock counts (no writes)
#   None             — do nothing
#
LOCK_ACTION = "stats"   # change to "lock_discards", "lock_keeps", or "unlock_all"
# ─────────────────────────────────────────────────────────────────────────────

if LOCK_ACTION == "lock_discards":
    triage_filter.lock_discards(cfg)
elif LOCK_ACTION == "lock_keeps":
    triage_filter.lock_keeps(cfg)
elif LOCK_ACTION == "unlock_all":
    triage_filter.unlock_all(cfg)
elif LOCK_ACTION == "stats":
    triage_filter.locked_stats(cfg)
else:
    print("LOCK_ACTION not set — nothing done.")


Triage lock status (13835 images total):
  Locked DISCARD : 0
  Locked KEEP    : 0
  Free (unlocked): 13,835


In [6]:
import json
import pandas as pd

triage_path = Path(cfg["paths"]["triage"])
summary_csv = triage_path / "triage_summary.csv"

if not summary_csv.exists():
    print("No triage_summary.csv found — run Cell 3 first.")
else:
    summary_df = pd.read_csv(summary_csv)

    total_images  = summary_df["total_images"].sum()
    total_flagged = summary_df["flagged_count"].sum()
    overall_ratio = total_flagged / max(1, total_images)

    print(f"Total images scored : {total_images}")
    print(f"Total flagged       : {total_flagged}")
    print(f"Overall flagged ratio: {overall_ratio:.1%}")
    print()

    # Show per-figure results for the 3 patents with the most flagged images
    top3 = summary_df.nlargest(3, "flagged_count")[["patent_id", "total_images", "flagged_count", "flagged_ratio"]]
    print("Top 3 patents by flagged count:")
    display(top3.reset_index(drop=True))
    print()

    for patent_id in top3["patent_id"]:
        json_path = triage_path / f"{patent_id}.json"
        if not json_path.exists():
            print(f"  {patent_id}: JSON not found")
            continue
        with open(json_path) as fh:
            data = json.load(fh)
        figures_df = pd.DataFrame(data["figures"])
        print(f"\n--- {patent_id}  (total={data['total']}  flagged={data['flagged']}) ---")
        display(figures_df[["file", "pred", "score_drawing", "score_table", "keep"]])


Total images scored : 13835
Total flagged       : 3418
Overall flagged ratio: 24.7%

Top 3 patents by flagged count:


,patent_id,total_images,flagged_count,flagged_ratio
0,CN116280179A_86786318,12,12,1.0
1,CN119099860A_93717638,12,12,1.0
2,CN120191516A_96063272,12,12,1.0




--- CN116280179A_86786318  (total=12  flagged=12) ---


,file,pred,score_drawing,score_table,keep
0,CN116280179APAFP.png,table,0.4875,0.5127,False
1,CN116280179A_202310171305.png,table,0.4875,0.5127,False
2,CN116280179A_HDA0004099463590000012.png,table,0.4956,0.5044,False
3,CN116280179A_HDA0004099463590000021.png,table,0.4949,0.5054,False
4,CN116280179A_HDA0004099463590000022.png,table,0.4944,0.5054,False
5,CN116280179A_HDA0004099463590000031.png,table,0.4910,0.5093,False
6,CN116280179A_HDA0004099463590000032.png,table,0.4834,0.5166,False
7,CN116280179A_HDA0004099463590000041.png,table,0.5054,0.4949,False
8,CN116280179A_HDA0004099463590000042.png,table,0.4878,0.5122,False
9,CN116280179A_HDA0004099463590000051.png,table,0.4890,0.5107,False



--- CN119099860A_93717638  (total=12  flagged=12) ---


,file,pred,score_drawing,score_table,keep
0,202411421431.png,table,0.5020,0.4980,False
1,CN119099860APAFP.png,table,0.5020,0.4980,False
2,FT_1.png,table,0.4910,0.5093,False
3,FT_10.png,table,0.4924,0.5078,False
4,FT_2.png,table,0.4880,0.5122,False
5,FT_3.png,table,0.5020,0.4980,False
6,FT_4.png,table,0.4871,0.5127,False
7,FT_5.png,table,0.4951,0.5049,False
8,FT_6.png,table,0.4858,0.5142,False
9,FT_7.png,table,0.4961,0.5039,False



--- CN120191516A_96063272  (total=12  flagged=12) ---


,file,pred,score_drawing,score_table,keep
0,CN120191516APAFP.png,table,0.4893,0.5107,False
1,CN120191516A_202411687048.png,table,0.4893,0.5107,False
2,CN120191516A_HDA0005149869780000021.png,table,0.4883,0.5117,False
3,CN120191516A_HDA0005149869780000031.png,table,0.4958,0.5044,False
4,CN120191516A_HDA0005149869780000041.png,table,0.4980,0.5020,False
5,CN120191516A_HDA0005149869780000051.png,table,0.4814,0.5186,False
6,CN120191516A_HDA0005149869780000052.png,table,0.4836,0.5166,False
7,CN120191516A_HDA0005149869780000053.png,table,0.4873,0.5127,False
8,CN120191516A_HDA0005149869780000061.png,table,0.4900,0.5098,False
9,CN120191516A_HDA0005149869780000062.png,table,0.4895,0.5103,False


## Cell 5 — Visual inspection: kept vs. discarded images

Shows the actual images for one patent so you can judge whether the threshold is working correctly.  
- **Green border** = kept (`keep=True`, `score_drawing ≥ threshold`)  
- **Red border** = discarded (`keep=False`)  

Change `INSPECT_PATENT` to any patent ID from the triage results, or leave it as `None` to auto-pick the first processed patent.

In [7]:
import json
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from PIL import Image

triage_path = Path(cfg["paths"]["triage"])
raw_dir     = Path(cfg["paths"]["raw_images"])
MAX_COLS    = 4

# All processed patents, sorted
patent_ids = sorted(p.stem for p in triage_path.glob("*.json")
                    if p.stem != "triage_summary")
if not patent_ids:
    print("No triage JSON files found — run Cell 3 first.")
    raise SystemExit

print(f"Found {len(patent_ids)} processed patents. Use the buttons or dropdown to navigate.")

# ── Widgets ───────────────────────────────────────────────────────────────────
idx_state   = {"i": 0, "updating": False}   # flag prevents dropdown re-entrancy
out         = widgets.Output()
btn_prev    = widgets.Button(description="◀ Prev", button_style="info",  layout=widgets.Layout(width="120px"))
btn_next    = widgets.Button(description="Next ▶", button_style="info",  layout=widgets.Layout(width="120px"))
lbl_pos     = widgets.Label(value=f"1 / {len(patent_ids)}")
patent_dd   = widgets.Dropdown(options=patent_ids, value=patent_ids[0],
                                layout=widgets.Layout(width="360px"))

def show_patent(patent_id):
    json_path = triage_path / f"{patent_id}.json"
    with open(json_path) as fh:
        data = json.load(fh)
    figures = data["figures"]
    n       = len(figures)
    if n == 0:
        print(f"{patent_id}: no figures in triage JSON")
        return
    n_kept  = sum(1 for f in figures if f["keep"])
    n_disc  = n - n_kept

    n_cols = min(MAX_COLS, n)
    n_rows = math.ceil(n / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 1.8, n_rows * 2.0),
                             squeeze=False, dpi=80)
    flat_axes = [ax for row in axes for ax in row]

    patent_dir = raw_dir / patent_id
    for ax, fig_info in zip(flat_axes, figures):
        img_path = patent_dir / fig_info["file"]
        try:
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "not found", ha="center", va="center",
                    color="grey", transform=ax.transAxes)
        keep  = fig_info["keep"]
        color = "#2ecc71" if keep else "#e74c3c"
        label = "KEEP" if keep else "DISCARD"
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(4)
        ax.set_title(f"{fig_info['file']}\n{label}  {fig_info['score_drawing']:.3f}",
                     fontsize=7, color=color, pad=3)
        ax.set_xticks([]); ax.set_yticks([])

    for ax in flat_axes[n:]:
        ax.axis("off")

    keep_patch    = mpatches.Patch(color="#2ecc71", label=f"Kept ({n_kept})")
    discard_patch = mpatches.Patch(color="#e74c3c", label=f"Discarded ({n_disc})")
    fig.legend(handles=[keep_patch, discard_patch], loc="lower center",
               ncol=2, fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"{patent_id}  —  {n_kept}/{n} kept  (threshold={THRESHOLD})",
                 fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

def refresh():
    i = idx_state["i"]
    # Update label and dropdown without triggering the observer callback
    lbl_pos.value = f"{i+1} / {len(patent_ids)}"
    idx_state["updating"] = True
    patent_dd.value = patent_ids[i]
    idx_state["updating"] = False
    with out:
        clear_output(wait=True)
        show_patent(patent_ids[i])

def on_prev(_):
    if idx_state["i"] > 0:
        idx_state["i"] -= 1
        refresh()

def on_next(_):
    if idx_state["i"] < len(patent_ids) - 1:
        idx_state["i"] += 1
        refresh()

def on_dropdown(change):
    # Only react to user-driven value changes, not programmatic ones from refresh()
    if change["name"] == "value" and not idx_state["updating"]:
        new_val = change["new"]
        if new_val in patent_ids:
            idx_state["i"] = patent_ids.index(new_val)
            refresh()

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
patent_dd.observe(on_dropdown)

nav_bar = widgets.HBox([btn_prev, lbl_pos, btn_next, patent_dd])
display(nav_bar, out)
refresh()

Found 1614 processed patents. Use the buttons or dropdown to navigate.


Output()

## Cell 6 — Review pending discards

Shows every image flagged as **discarded** that has **not yet been locked** (i.e. not reviewed in a previous session).

- **Red border** = DISCARD — will be permanently locked on Commit
- **Green border** = APPROVED — will be flipped to KEEP on Commit

Click any image to toggle between DISCARD and APPROVED.  Nothing is written to disk until you run the **Commit cell** below.

> Already-locked images (from previous sessions) are hidden — run `reset_review` in the Commit cell to bring them back.

In [3]:
import importlib, triage_filter
importlib.reload(triage_filter)

import json, math, threading
import matplotlib
matplotlib.use("widget")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from PIL import Image

triage_path = Path(cfg["paths"]["triage"])
raw_dir     = Path(cfg["paths"]["raw_images"])

PAGE_SIZE = 200
MAX_COLS  = 5

# ── Load non-locked discarded images only ─────────────────────────────────────
# Images already locked (reviewed in a previous session) are hidden.
def load_pending_discards():
    result = []
    for json_path in sorted(triage_path.glob("*.json")):
        if json_path.stem == "triage_summary":
            continue
        with open(json_path) as fh:
            data = json.load(fh)
        for fig in data["figures"]:
            if not fig["keep"] and not fig.get("locked"):
                result.append({
                    "patent_id":     json_path.stem,
                    "file":          fig["file"],
                    "score_drawing": fig["score_drawing"],
                    "score_table":   fig["score_table"],
                })
    return result

discarded = load_pending_discards()

# approved: images the user clicked to flip to KEEP this session (not yet on disk)
approved  = set()

# visited_pages: page indices actually rendered (shown) this session. Commit
# only locks images belonging to these pages — pages you never scroll to stay
# pending instead of being silently locked as DISCARD by default.
visited_pages = set()

n_total = len(discarded)
n_pages = max(1, math.ceil(n_total / PAGE_SIZE))
print(f"Pending review (non-locked discards): {n_total}  ({n_pages} pages of {PAGE_SIZE})")
print("  Red border  = DISCARD  (will be locked on Commit)")
print("  Green border = APPROVED (will be set to KEEP on Commit)")
print("  Click any image to toggle it to APPROVED.")
print("  Only pages you actually view here get committed — unvisited pages stay pending.")
print()
print("When done reviewing, run the LAST CELL to commit your decisions to disk.")

# ── Widgets ───────────────────────────────────────────────────────────────────
page_state = {"p": 0, "fig": None, "flat_axes": [], "batch": []}
out_disc   = widgets.Output()
btn_dp     = widgets.Button(description="◀ Prev", button_style="warning", layout=widgets.Layout(width="120px"))
btn_dn     = widgets.Button(description="Next ▶", button_style="warning", layout=widgets.Layout(width="120px"))
lbl_dp     = widgets.Label(value=f"Page 1 / {n_pages}")
lbl_info   = widgets.Label(value="")
lbl_counts = widgets.Label(value=f"Approved: 0 / {n_total}")
btn_commit = widgets.Button(description="Commit Page & Next", button_style="success", layout=widgets.Layout(width="180px"))
committed_pages = set()

def _ax_color(entry):
    key = (entry["patent_id"], entry["file"])
    return ("#2ecc71", "APPROVED ✓") if key in approved else ("#e74c3c", "DISCARD")

def show_page(p):
    batch  = discarded[p * PAGE_SIZE : (p + 1) * PAGE_SIZE]
    n      = len(batch)
    if n == 0:
        return
    visited_pages.add(p)
    n_cols = min(MAX_COLS, n)
    n_rows = math.ceil(n / max(n_cols, 1))

    if page_state["fig"] is not None:
        plt.close(page_state["fig"])

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 3.2, n_rows * 3.4),
                             squeeze=False, dpi=80)
    flat_axes = [ax for row in axes for ax in row]
    page_state.update({"fig": fig, "flat_axes": flat_axes, "batch": batch})

    for ax, entry in zip(flat_axes, batch):
        img_path = raw_dir / entry["patent_id"] / entry["file"]
        try:
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "not found", ha="center", va="center",
                    color="grey", transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        color, label = _ax_color(entry)
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(4)
        ax.set_title(
            f"{entry['patent_id']}\n{entry['file']}\n{label}  draw={entry['score_drawing']:.3f}",
            fontsize=6, color=color, pad=3
        )

    for ax in flat_axes[n:]:
        ax.axis("off")

    fig.suptitle(
        f"Pending discards — page {p+1}/{n_pages}  "
        f"(images {p*PAGE_SIZE+1}–{p*PAGE_SIZE+n} of {n_total})  "
        f"| Click image → APPROVED",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()

    def on_click(event):
        if event.inaxes is None:
            return
        fa, ba = page_state["flat_axes"], page_state["batch"]
        for i, ax in enumerate(fa[:len(ba)]):
            if ax is event.inaxes:
                entry = ba[i]
                key   = (entry["patent_id"], entry["file"])
                # toggle
                if key in approved:
                    approved.discard(key)
                else:
                    approved.add(key)
                color, label = _ax_color(entry)
                for spine in ax.spines.values():
                    spine.set_edgecolor(color)
                    spine.set_linewidth(4)
                ax.set_title(
                    f"{entry['patent_id']}\n{entry['file']}\n{label}  draw={entry['score_drawing']:.3f}",
                    fontsize=6, color=color, pad=3
                )
                fig.canvas.draw_idle()
                lbl_info.value   = f"{'✓ APPROVED' if key in approved else '✗ back to DISCARD'}: {entry['patent_id']} / {entry['file']}"
                lbl_counts.value = f"Approved: {len(approved)} / {n_total}"
                return

    fig.canvas.mpl_connect("button_press_event", on_click)
    plt.show()

def refresh_disc(new_p):
    page_state["p"] = new_p
    lbl_dp.value    = f"Page {new_p + 1} / {n_pages}"
    lbl_info.value  = ""
    with out_disc:
        clear_output(wait=True)
        show_page(new_p)

def commit_current_page(_):
    p = page_state["p"]
    batch = page_state["batch"]
    if not batch:
        return
    stats = triage_filter.commit_batch(cfg, batch, approved)
    committed_pages.add(p)
    lbl_info.value = (
        f"✓ Page {p+1} committed to disk — "
        f"{stats['newly_kept']} kept, {stats['newly_locked_discard']} discarded."
    )
    if p < n_pages - 1:
        refresh_disc(p + 1)
    else:
        with out_disc:
            clear_output(wait=True)
            print("Last page — all reviewed pages committed.")

btn_commit.on_click(commit_current_page)
btn_dp.on_click(lambda _: refresh_disc(page_state["p"] - 1) if page_state["p"] > 0 else None)
btn_dn.on_click(lambda _: refresh_disc(page_state["p"] + 1) if page_state["p"] < n_pages - 1 else None)

display(widgets.HBox([btn_dp, lbl_dp, btn_dn, btn_commit, lbl_counts, lbl_info]), out_disc)
refresh_disc(0)


Pending review (non-locked discards): 18  (1 pages of 200)
  Red border  = DISCARD  (will be locked on Commit)
  Green border = APPROVED (will be set to KEEP on Commit)
  Click any image to toggle it to APPROVED.
  Only pages you actually view here get committed — unvisited pages stay pending.

When done reviewing, run the LAST CELL to commit your decisions to disk.


Output()

## Cell 7 — Commit review decisions

Writes your Cell-6 review session to disk.

| `COMMIT_ACTION` | Effect |
|-----------------|--------|
| `"commit"` | Lock every APPROVED image as KEEP, lock every remaining DISCARD as discarded. They won't appear in the viewer again. |
| `"reset_review"` | Remove all discard locks — images reappear in Cell 6 for another review pass. Locked KEEPs are preserved. |
| `"stats"` | Show current lock counts (no writes). |

> **Typical session:**  
> 1. Run Cell 6, flip the false-positives to APPROVED  
> 2. Run this cell with `COMMIT_ACTION = "commit"`  
> 3. Next time you open the notebook, only *new* unreviewed discards show up  

> **To start fresh** (e.g. after changing threshold):  
> Set `COMMIT_ACTION = "reset_review"`, then re-run Cell 3b with `MODE = "reset"`

In [9]:
import importlib, triage_filter
importlib.reload(triage_filter)

# ── Choose action ─────────────────────────────────────────────────────────────
#
#   "commit"        — write this session's decisions to disk (only pages you
#                     actually viewed in Cell 6 above — see reviewed_keys below)
#   "reset_review"  — unlock discard locks so they reappear in the viewer
#   "stats"         — show lock counts only (no writes)
#
COMMIT_ACTION = "commit"
# ─────────────────────────────────────────────────────────────────────────────

if COMMIT_ACTION == "commit":
    # `approved` and `visited_pages` are built up in Cell 6 above.
    # Only images on pages you actually viewed get committed — anything you
    # never scrolled to stays pending instead of being silently locked as
    # discard. If Cell 6 wasn't run this session, both default to empty so
    # this commit is a safe no-op rather than mass-locking everything pending.
    try:
        _approved = approved
        _visited_pages = visited_pages
    except NameError:
        _approved = set()
        _visited_pages = set()
        print("⚠  Cell 6 was not run this session — nothing to commit.")

    _reviewed_keys = {
        (fig["patent_id"], fig["file"])
        for p in _visited_pages
        for fig in discarded[p * PAGE_SIZE : (p + 1) * PAGE_SIZE]
    }
    triage_filter.commit_review(cfg, _approved, reviewed_keys=_reviewed_keys)

elif COMMIT_ACTION == "reset_review":
    triage_filter.reset_review(cfg)

elif COMMIT_ACTION == "stats":
    triage_filter.locked_stats(cfg)

else:
    print(f"Unknown COMMIT_ACTION={COMMIT_ACTION!r} — choose commit / reset_review / stats")


commit_review: 0 images flipped to KEEP (locked).
               3418 discards locked permanently.
Summary rewritten: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/triage/triage_summary.csv
